In [0]:
%sql
select * from employeeproject.default.employee;

In [0]:
describe employeeproject.default.employee

In [0]:
%sql
describe employeeproject.default.employee

In [0]:

from delta.tables import DeltaTable
target = "employeeproject.default.employee"
delta_table = DeltaTable.forName(spark, target)
display(
    delta_table.history().select("version", "timestamp", "operation")
)

In [0]:

from delta.tables import DeltaTable
target = "employeeproject.default.employee"
delta_table = DeltaTable.forName(spark, target)
display(
    delta_table.history().select("version", "timestamp", "operation")
)

In [0]:
%sql
SELECT version, timestamp, operation
FROM table(employeeproject.default.employee.history())

In [0]:
df = spark.read.option("versionAsof", "0").table("employeeproject.default.employee")
display(df)

In [0]:
from pyspark.sql.functions import col
v0_timestamp = delta_table.history() \
    .filter(col("version") == 0) \
    .select("timestamp") \
    .collect()[0][0]
print(f"Reading data as of timestamp: {v0_timestamp}")

In [0]:
display(v0_timestamp)

In [0]:
df_ts = spark.read \
    .format("delta") \
    .option("timestampAsOf", str(v0_timestamp)) \
    .table(target)
df_ts.show()

In [0]:
print("Restoring table to Version 0:")
spark.sql(f"""RESTORE TABLE {target} TO VERSION AS OF 0""")
spark.sql(f"""SELECT * FROM {target}""").show()
print("Table restored to original state (Version 0).")

In [0]:
from delta.tables import DeltaTable
target = "employeeproject.default.employee"
delta_table = DeltaTable.forName(spark, target)
display(
    delta_table.history().select("version", "timestamp", "operation")
)

In [0]:
target_new = "employeeproject.default.employee_partitioned"
df_ts.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("department") \
    .saveAsTable(f"{target_new}")

In [0]:
display(df_ts).sort("city")

### object path
user/hive/warehouse/catalog/schema/employee_partitioned

patition names:
department=Engineering
department=Marketing
department=Finance
department=HR

In [0]:
spark.sql(f"""
    OPTIMIZE {target_new}
    ZORDER BY (city)
""")